# EA3 — Proyecto final: procesamiento distribuido y cierre del caso

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *GRUPO 29* |
| **Integrantes** | *ZHARICK ELISA GARCIA TONCEL · DARIO ANDRES GUERRERO CANTILLO |
| **Caso de estudio** | *TABLA clickstream* |
| **Fecha de entrega** | Domingo 27 de Septiembre |
| **🎥 Enlace al video** | *https://drive.google.com/file/d/17-aOypRrC5yxtbTiJowvXs8BQ1km93DR/view?usp=sharing* |
|  **Link Repositorio Github**|https://github.com/DarioAGC39/bigdata-2026b-GRUPO29/tree/main    |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta
> a la propia. Un enlace inaccesible se califica como no entregado.

## Cómo usar esta plantilla

Las celdas en *cursiva* son instrucciones: **reemplácenlas por su contenido.**

| Sección | Qué va ahí | Puntos |
|---|---|---|
| 1 · Contexto | De dónde vienen, en una página | — |
| 2 · Capa oro | La tabla de consumo y el recorrido del dato | 20 |
| 3 · Medición inicial | El plan de ejecución y los tres tiempos | 15 |
| 4 · Optimización | La técnica, el plan nuevo y la comparación | 20 |
| 5 · Límites | Cuándo esa técnica sería contraproducente | 10 |
| 6 · Cierre del caso | La pregunta orientadora y las conclusiones | 10 |
| 7 · Reparto y uso de IA | Quién hizo qué | — |
| ⭐ Opcional | Tablero o Genie sobre la capa oro | +5 |

Los 25 puntos restantes son individuales y se evalúan en el video.

# 1 · Contexto



La empresa Wanderbricks tiene  datos relacionados con las interacciones o movimientos 
realizadas por los usuarios dentro de su plataforma. Para este proyecto
trabajamos principalmente con la tabla clickstream, que tiene información
sobre los  eventos, usuarios, propiedades, fecha y hora, dispositivo y origen de
las visitas.

En la actividad  EA1 hicimos el análisis de manera  inicial de los datos y se construyeron las
capas Bronce y Silver. En la actividad EA2 se organizó el entorno en Databricks, se
definieron permisos, se revisó el linaje y se automatizaron procesos mediante
Jobs. En esta actividad EA3 se completa el pipeline con una capa Gold orientada al
consumo y se analiza el rendimiento de una operación distribuida en Spark.

La pregunta de negocio que se buscamos responder a traves de la capa Gold es:
**¿Qué tipos de eventos realizan los usuarios en Wanderbricks y desde qué
dispositivo y fuente de acceso se generan con mayor frecuencia?**

# 2 · Capa oro · 20 puntos

## 2.1 · Qué pregunta responde

¿Qué tipos de eventos realizan los usuarios en Wanderbricks y desde qué dispositivo y fuente de acceso se generan con mayor frecuencia?



In [0]:
%sql
CREATE OR REPLACE TABLE bigdata_grupo29.wanderbricks.gold_clickstream AS
SELECT
    event AS tipo_evento,
    device AS dispositivo,
    referrer AS fuente_acceso,
    COUNT(*) AS total_eventos
FROM bigdata_grupo29.wanderbricks.silver_clickstream
GROUP BY event, device, referrer
ORDER BY total_eventos DESC;


num_affected_rows,num_inserted_rows


**Comprobamos la capa de Oro y La tabla resultante**

In [0]:
%sql
SELECT * 
FROM bigdata_grupo29.wanderbricks.gold_clickstream;

tipo_evento,dispositivo,fuente_acceso,total_eventos
click,desktop,google,2193
search,tablet,direct,2180
view,desktop,email,2155
search,tablet,ad,2147
filter,tablet,direct,2136
click,tablet,email,2135
search,desktop,email,2134
view,tablet,email,2133
click,tablet,ad,2133
filter,tablet,email,2131


## 2.3 · El recorrido del dato

*Cuántos registros hay en cada capa y a qué corresponde cada diferencia. Si algo se
descartó, digan por qué.*

In [0]:
%sql
SELECT
    COUNT(*) AS filas_gold,
    SUM(total_eventos) AS eventos_totales
FROM bigdata_grupo29.wanderbricks.gold_clickstream;

filas_gold,eventos_totales
48,100000


**El recorrido de los datos se hizo de la siguiente manera**:

- **Bronce:** 100000 registros. tiene los datos almacenados del inicio.
- **Plata:** 100000 registros. Mantiene todos los registros, pero transforma la
  información semiestructurada de metadata en las columnas device y referrer
- **Oro:** 48 registros. Estos registros son de las diferentes
  combinaciones de tipo de evento, dispositivo y fuente de acceso.

La diferencia entre plata y Oro no corresponde a datos descartados. Los
100000 eventos los  agrupamos con GROUP BY, generando 48
combinaciones y calculando para cada una el número total de eventos.

Es decir que, la capa oro reduce la cantidad de filas porque pone la
información de forma resumida y orientada al análisis.

# 3 · Medición inicial · 15 puntos

## 3.1 · La operación que vamos a optimizar



In [0]:
# Función de medición — úsenla tal cual, no la modifiquen
def medir(descripcion, funcion, repeticiones=3):
    """Calienta, repite y reporta la mediana. Las tres mediciones quedan visibles."""
    funcion()                                   # calentamiento, se descarta
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempos.append(time.time() - inicio)

    mediana = statistics.median(tiempos)
    print(f"{descripcion}")
    print(f"   mediciones : {', '.join(f'{t:.2f}s' for t in tiempos)}")
    print(f"   MEDIANA    : {mediana:.2f}s")
    print(f"   variación  : {max(tiempos)-min(tiempos):.2f}s\n")
    return mediana

**Operación seleccionada**

Para evaluar el rendimiento seleccionamos una operación de JOIN entre las
capas Bronce y Plata.

La consulta relaciona los eventos mediante el campo** user_id** Se eligió
esta operación porque un JOIN necesita intercambio y procesamiento de datos
entre los nodos de Spark, por lo que nos permite observar el comportamiento del
plan de ejecución y medir el efecto de una optimización.

La técnica de optimización que  seleccionamos será **BROADCAST**, para evitar
movimientos innecesarios de una tabla pequeña entre los nodos.

## 3.2 · El plan de ejecución inicial

*Ejecuten `.explain()` y dejen la salida visible. Después señalen dónde está el costo.*

In [0]:
bronze = spark.table("bigdata_grupo29.wanderbricks.bronze_clickstream")
silver = spark.table("bigdata_grupo29.wanderbricks.silver_clickstream")

consulta = (
    bronze.alias("b")
    .join(silver.alias("s"), "user_id", "inner")
    .hint("merge")
    .select("b.user_id")
)

consulta.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [user_id#11772L]
         +- PhotonBroadcastHashJoin [user_id#11772L], [user_id#11784L], Inner, BuildRight, false, true, false
            :- PhotonScan parquet bigdata_grupo29.wanderbricks.bronze_clickstream[user_id#11772L] DataFilters: [isnotnull(user_id#11772L)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-cetfp/uc/449ac041-bef6-4f63-8740-3733df112db0..., OptionalDataFilters: [hashedrelationcontains(user_id#11772L)], PartitionFilters: [], ReadSchema: struct<user_id:bigint>, RequiredDataFilters: [isnotnull(user_id#11772L)]
            +- PhotonShuffleExchangeSource false
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#7811]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonScan parquet bigdata_grupo29.wanderbricks.silver_c

In [0]:
consulta = (
    bronze.alias("b")
    .join(silver.alias("s"), "user_id")
    .select("b.user_id")
)

consulta.count()

179952

In [0]:
import time

tiempos = []

# Calentamiento
consulta.count()

# 3 mediciones
for i in range(3):
    inicio = time.perf_counter()
    consulta.count()
    fin = time.perf_counter()
    tiempos.append(fin - inicio)

print("Tiempos:", tiempos)
print("Mediana:", sorted(tiempos)[1])

Tiempos: [0.960262658999909, 0.9690705590001016, 0.8979177249998429]
Mediana: 0.960262658999909


In [0]:
from pyspark.sql.functions import broadcast

consulta_opt = (
    bronze.alias("b")
    .join(broadcast(silver.alias("s")), "user_id")
    .select("b.user_id")
)

consulta_opt.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [user_id#12246L]
         +- PhotonBroadcastHashJoin [user_id#12246L], [user_id#12251L], Inner, BuildRight, false, true, false
            :- PhotonScan parquet bigdata_grupo29.wanderbricks.bronze_clickstream[user_id#12246L] DataFilters: [isnotnull(user_id#12246L)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-cetfp/uc/449ac041-bef6-4f63-8740-3733df112db0..., OptionalDataFilters: [hashedrelationcontains(user_id#12246L)], PartitionFilters: [], ReadSchema: struct<user_id:bigint>, RequiredDataFilters: [isnotnull(user_id#12246L)]
            +- PhotonShuffleExchangeSource false
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#9030]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonScan parquet bigdata_grupo29.wanderbricks.silver_c

## 3.3 · Los tres tiempos

In [0]:
import time

tiempos_opt = []

# Calentamiento
consulta_opt.count()

# 3 mediciones
for i in range(3):
    inicio = time.perf_counter()
    consulta_opt.count()
    fin = time.perf_counter()
    tiempos_opt.append(fin - inicio)

print("Tiempos optimizados:", tiempos_opt)
print("Mediana optimizada:", sorted(tiempos_opt)[1])


**Explicacion plan de mejora no visible:**

La ejecución inicial presentó una mediana de **0,9603 segundos**. Después de aplicar explícitamente **broadcast()**`, la mediana fue de **0,9380 segundos**.

Aunque se observa una pequeña reducción de  **0,02 segundos**, la diferencia es muy baja y los tres tiempos tienen variaciones entre ejecuciones. entonces, no se considera una mejora  del rendimiento.



# 4 · Optimización · 20 puntos

## 4.1 · Qué técnica aplicamos y por qué

*Una sola técnica, bien elegida. Expliquen por qué esa y no otra, en función de lo que
vieron en el plan inicial.*

In [0]:
from pyspark.sql.functions import broadcast

# Aplicar la optimización con broadcast
consulta_opt = (
    bronze.alias("b")
    .join(broadcast(silver.alias("s")), "user_id")
    .select("b.user_id")
)

# Mostrar el nuevo plan de ejecución
consulta_opt.explain()


## 4.2 · El plan de ejecución resultante

In [0]:
bronze = spark.table("bigdata_grupo29.wanderbricks.bronze_clickstream")
silver = spark.table("bigdata_grupo29.wanderbricks.silver_clickstream")

consulta = (
    bronze.alias("b")
    .join(silver.alias("s"), "user_id", "inner")
    .hint("merge")
    .select("b.user_id")
)

consulta.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [user_id#11897L]
         +- PhotonBroadcastHashJoin [user_id#11897L], [user_id#11902L], Inner, BuildRight, false, true, false
            :- PhotonScan parquet bigdata_grupo29.wanderbricks.bronze_clickstream[user_id#11897L] DataFilters: [isnotnull(user_id#11897L)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-cetfp/uc/449ac041-bef6-4f63-8740-3733df112db0..., OptionalDataFilters: [hashedrelationcontains(user_id#11897L)], PartitionFilters: [], ReadSchema: struct<user_id:bigint>, RequiredDataFilters: [isnotnull(user_id#11897L)]
            +- PhotonShuffleExchangeSource false
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#7888]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonScan parquet bigdata_grupo29.wanderbricks.silver_c

**EXPLICACION:**
Después de que aplicamos explícitamente la función **broadcast()**`  se volvió a capturar el plan de ejecución mediante **explain().
**
El plan resultante se siguio utilizando la operación **PhotonBroadcastHashJoin**`, indicando que el motor Photon de Databricks ya estaba aplicando automáticamente esta estrategia antes de la optimización manual.

No se observaron cambios que fuesen significativos entre el plan inicial y el plan resultante. Esto nos  explica por qué los tiempos de ejecución obtenidos antes y después de la optimización fueron muy similares.

La principal operación identificada en ambos planes fue:

**- `PhotonBroadcastHashJoin**`

Esto indica que Spark distribuye una de las tablas a los ejecutores para evitar operaciones de intercambio más costosas durante el JOIN.

## 4.3 · Qué cambió entre los dos planes

*Comparen los dos planes de forma explícita. No basta con poner los dos números: hay que
decir qué operación desapareció o se redujo y por qué eso baja el costo.*

| | Antes | Después |
|---|---|---|
| Estrategia de cruce |0,9603 |0,9380 |
| Movimiento de datos |0,9691 |1,0973  |
| Ordenamiento |0,8979 |0,8204 |
| **Mediana**  | **0,9603**    |**0,9380**       |

**Conclusión de la optimización:**

*Qué hacía antes, qué cambió en el plan, y el número. En ese orden.*

> 💡 **Si no mejoró nada, repórtenlo igual** y expliquen por qué: volumen insuficiente,
> Spark ya lo estaba resolviendo solo, u otra razón. Una medición honesta vale más que un
> número inflado.

**analisis obtenidos**
La consulta original presentó una mediana de **0,9603 segundos**, mientras que la consulta optimizada presentó una mediana de **0,9380 segundos**.

La diferencia observada fue de  **0,02 segundos**, por lo que no representa una mejora significativa del rendimiento.

Al comparar los planes de ejecución antes y después de la optimización, se pudo ver  que ambos utilizaron la operación **PhotonBroadcastHashJoin**. Esto quiere decir  que Databricks Photon ya estaba aplicando automáticamente una estrategia de Broadcast Join antes de que se agregara explícitamente la función **`broadcast()`.**

dado esto, la optimización manual no produjo cambios muy notables en el plan físico ni en los tiempos de ejecución. El motor ya había seleccionado una estrategia eficiente para ejecutar la consulta.

Este resultado demuestra la importancia de analizar primero el plan de ejecución antes de aplicar optimizaciones manuales, ya que en algunos casos el motor de procesamiento ya implementa automáticamente la mejor estrategia que este disponible o que podamos usar.

## 4.4 · Los tiempos, comparados

**optimizacion con datos contraproducentes**

El plan de ejecución tampoco presentó un cambio importante: tanto antes como después se utilizó PhotonBroadcastHashJoin`. Esto quiere decir que Databricks/Photon ya estaba aplicando automáticamente esta estrategia de optimización.

En este caso, la optimización manual con broadcast() no dio un beneficio relevante. Ademas el uso de broadcast()` puede ser contraproducente cuando la tabla que estamos intentando enviar a los nodos es muy grande. En este caso puede aumentar el consumo de memoria y afectar el rendimiento.

También esto no aporta beneficios cuando el motor de Spark o Photon ya mira automáticamente la mejor estrategia, como ocurrió en esta prueba.


# 5 · Límites de la optimización · 10 puntos

*¿En qué condiciones esta misma técnica dejaría de servir, o incluso empeoraría el resultado?*

*Toda técnica tiene un rango donde ayuda y otro donde estorba. Identificar ese límite es lo
que demuestra que entendieron el mecanismo y no solo el procedimiento.*

*Ata su respuesta a su caso: qué tendría que cambiar en sus datos para que esta optimización
dejara de tener sentido.*

 **Límites de la optimización**

La optimización evaluada en este proyecto fue el uso explícito de **broadcast()** en una operación de JOIN. Sin embargo, los resultados mostraron que la mejora obtenida fue mínima, ya que Databricks Photon ya estaba utilizando automáticamente un **PhotonBroadcastHashJoin`**.

Esto nos deja ver  que una optimización no siempre genera beneficios adicionales. En algunos casos, el motor de procesamiento ya selecciona la estrategia más eficiente y cualquier intervención manual produce cambios poco significativos.

Además, el uso de `**broadcast()**` tiene  limitaciones cuando la tabla que se desea distribuir es demasiado grande. En ese escenario, cada ejecutor debe mantener una copia de la tabla en memoria, aumentando el consumo de recursos y pudiendo afectar negativamente el rendimiento.

Por esta razón, antes de aplicar una optimización es necesario analizar el plan de ejecución y verificar si el motor ya está utilizando una estrategia similar. 

# 6 · Cierre del caso · 10 puntos

## 6.1 · La pregunta orientadora

> Una empresa recibe datos de su portal web, de Facebook, de Instagram, de TikTok y de
> WhatsApp Business. **¿Bajo qué paradigma, con qué herramientas y con qué arquitectura
> debería analizarlos?**

*Un par de párrafos, no un ensayo. La respuesta debe apoyarse en lo que construyeron
durante el curso, no en generalidades.*

** pregunta orientadora**

**Una empresa que recibe datos desde su portal web, Facebook, Instagram, TikTok y WhatsApp Business**

**R/** debería utilizar un paradigma de procesamiento distribuido, ya que se trata de múltiples fuentes que pueden generar grandes volúmenes de datos con diferentes estructuras y velocidades de llegada. Este enfoque permite almacenar, procesar e integrar la información de manera escalable para apoyar la toma de decisiones.

A partir de lo desarrollado durante el curso, una alternativa adecuada sería utilizar Databricks como plataforma de análisis, apoyada en Apache Spark para el procesamiento distribuido y Delta Lake para el almacenamiento de los datos. La arquitectura recomendada es la de capas Bronce, plata y Oro. En **Bronce** se almacenan los datos originales provenientes de todas las fuentes, en **Plata** se realizan las transformaciones y procesos de calidad de datos, y en **Oro** se generan tablas agregadas y orientadas al negocio. Esta arquitectura facilita el gobierno de los datos, el análisis posterior y la construcción de reportes o tableros para los usuarios finales.

## 6.2 · Conclusiones del proyecto

*Qué funcionó, qué no, y qué harían distinto si empezaran de nuevo. Deben derivarse de sus
resultados, no de impresiones generales sobre la herramienta.*

 

A lo largo del proyecto se logró construir un pipeline completo utilizando Databricks y la arquitectura Bronce, Plata y Oro. Se trabajó con la tabla de **clickstream**, comenzando con el análisis y validación de la calidad de los datos, continuando con la organización y gobierno del entorno, y finalizando con la construcción de una capa Oro orientada al consumo por parte del negocio.

Uno de los principales resultados fue comprobar cómo la arquitectura por capas permite mantener un flujo ordenado de transformación de datos. Los 100000 registros iniciales se conservaron durante las etapas de procesamiento y posteriormente fueron resumidos en una tabla Oro de 48 combinaciones agregadas, facilitando el análisis sin perder información.

En la etapa de optimización se observó que Databricks Photon ya aplicaba automáticamente estrategias eficientes de ejecución, como **PhotonBroadcastHashJoin`**. Esto permitió comprender que no todas las optimizaciones manuales generan mejoras significativas y que la revisión del plan de ejecución es un paso fundamental antes de modificar una consulta.

Si el proyecto se desarrollara nuevamente desde el inicio, sería recomendable trabajar con volúmenes de datos más grandes y con un mayor número de tablas relacionadas para observar con mayor claridad el impacto de diferentes técnicas de optimización y procesamiento distribuido.

# 7 · Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| *Zharick Elisa Garcia Toncel* |primer parte del trabajo | |
| *(Dario Andres Guerrero Cantillo)* |Segunda parte del trabajo | |


**Uso de asistentes de IA:**

*Declaren dónde los usaron. Está permitido; lo que se evalúa es que puedan explicar
cualquier línea del código en el video.*

---
# ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] La capa oro existe y sus columnas se entienden sin explicación
- [ ] Las **tres mediciones** están visibles, antes y después, con la mediana
- [ ] Los **dos planes de ejecución** están en el notebook, con la salida visible
- [ ] Está escrito por qué mejoró y cuándo la técnica sería contraproducente
- [ ] La pregunta orientadora está respondida, apoyada en lo que construyeron
- [ ] No quedaron textos en cursiva de esta plantilla sin reemplazar
- [ ] La sección 7 coincide con lo que cada uno sustenta en el video
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] El `.ipynb` está confirmado en el repositorio, en `/ea3`
- [ ] El HTML con salidas visibles está subido a Canvas

### ⚠️ Sobre la cuota

Este proyecto pide ejecutar la misma consulta **ocho veces** —calentamiento y tres
mediciones, dos veces—. Eso gasta bastante más de lo habitual.
**No dejen las mediciones para la última noche.**

---

**Entrega: domingo 27 de Septiembre, 11:59 p. m.**